In [1]:
import polars as pl
import polars.selectors as cs
import plotly.express as px

In [2]:
df = pl.read_parquet("../data/processed/clean_tripdata.parquet")
df.glimpse()

Rows: 6006904
Columns: 26
$ ride_id                       <str> 'B7516B7B2708CC24', 'BCF70C0B42130B92', 'E004845CDF26C9D4', 'CE6ACBC2A319AA59', '12F3FE6984F93383', '6B3160CE448781BA', 'FB10CC36BED3C86B', '4D47190D0D5E6E55', '02531D76F32E6F6B', '20E8890ED8FCBF1B'
$ rideable_type                 <str> 'classic_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'electric_bike', 'electric_bike', 'electric_bike', 'classic_bike', 'classic_bike', 'electric_bike'
$ started_at           <datetime[μs]> 2025-10-05 15:26:37.782000, 2026-09-11 02:06:11.044000, 2025-10-14 17:05:58.193000, 2026-05-27 10:27:11.879000, 2026-08-24 05:37:01.378000, 2025-10-04 11:53:40.309000, 2026-05-18 12:49:36.303000, 2026-05-24 12:27:14.893000, 2026-06-02 14:15:13.902000, 2026-07-26 15:41:55.611000
$ ended_at             <datetime[μs]> 2025-10-05 15:29:47.109000, 2026-09-11 02:10:50.196000, 2025-10-14 17:14:50.362000, 2026-05-27 10:29:47.582000, 2026-08-24 05:40:45.344000, 2025-10-04 12:15:08.315000, 2026-05-18 1

In [3]:
df.select(cs.numeric()).describe()

statistic,start_lat,start_lng,end_lat,end_lng,ride_length,day_of_week,start_hour,end_hour,month,distance_km
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6,6.006904e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",41.904664,-87.646991,41.905052,-87.647359,855.390892,4.009267,14.040382,14.182858,6.993687,2.247817
"""std""",0.044228,0.0269,0.044428,0.027177,1612.84099,1.949833,5.034889,5.086577,2.59172,1.975599
"""min""",41.64,-87.91,41.51,-88.1,60.0,1.0,0.0,0.0,1.0,0.0
"""25%""",41.88281,-87.66,41.882906,-87.660141,334.0,2.0,10.0,11.0,5.0,0.969966
"""50%""",41.9,-87.6429,41.9,-87.643353,566.0,4.0,15.0,15.0,7.0,1.654917
"""75%""",41.93,-87.63,41.930585,-87.63,977.0,6.0,18.0,18.0,9.0,2.91763
"""max""",42.07,-87.52,42.19,-87.37,86398.0,7.0,23.0,23.0,12.0,48.370801


In [4]:
# seconds -> minutes
df = df.with_columns(pl.col("ride_length").truediv(60).round(2))

# Station columns used for the null-rate metrics
STATION_COLS = ["start_station_name", "end_station_name"]

In [5]:
df["ride_length"].quantile([.75, .9, .99, .999])

[16.28, 27.17, 79.03, 245.53]

In [6]:
df["ride_length"].std() / df["ride_length"].mean()

1.885501334266289

In [7]:
fig = px.histogram(
    df,
    x="ride_length",
    nbins=60,
    labels={"ride_length": "Ride Length in Minutes"},
)
fig.show()

In [8]:
def profile(d: pl.DataFrame) -> pl.DataFrame:
    """Per-column summary: dtype, nulls (count and %), and unique values."""
    nulls = d.null_count().row(0)
    return pl.DataFrame({
        "column": d.columns,
        "dtype": [str(t) for t in d.dtypes],
        "nulls": nulls,
        "pct_null": [round(n / d.height * 100, 1) for n in nulls],
        "uniques": [d[c].n_unique() for c in d.columns],
    })


# Available statistics for continuous metrics (name -> expression builder)
STATS = {
    "avg": lambda c: c.mean(),
    "med": lambda c: c.median(),
    "std": lambda c: c.std(),
    "min": lambda c: c.min(),
    "max": lambda c: c.max(),
    "p90": lambda c: c.quantile(0.90),
    "p99": lambda c: c.quantile(0.99),
}


def summarize_by(
    d: pl.DataFrame,
    by: str,
    metrics: tuple[str, ...] = ("ride_length", "distance_km"),
    stats: tuple[str, ...] = ("avg", "med", "std"),
    share: bool = False,
    station_nulls: bool = False,
    digits: int = 2,
) -> pl.DataFrame:
    """Statistics of CONTINUOUS metrics (e.g. ride_length), grouped by `by`.

    stats: any keys of STATS.
    share: adds the group's % of total rides.
    station_nulls: adds the % of null stations WITHIN each group.
    """
    aggs = []
    if share:
        aggs.append(pl.len().mul(100 / d.height).alias("rides_pct"))
    for m in metrics:
        aggs += [STATS[s](pl.col(m)).round(digits).alias(f"{m}_{s}") for s in stats]
    if station_nulls:
        aggs += [
            pl.col(c).is_null().mean().mul(100).round(digits).alias(f"{c}_null_pct")
            for c in STATION_COLS
        ]
    return d.group_by(by).agg(aggs).sort(by)


def distribution_by(
    d: pl.DataFrame,
    by: str,
    col: str,
    pivot: bool = True,
    digits: int = 2,
) -> pl.DataFrame:
    """% of rides of each `by` group falling in each value of `col`.

    Each `by` group sums to 100%, so groups of different sizes are comparable.
    pivot=True: one row per `col` value, one column per `by` group.
    """
    out = (
        d.group_by(by, col)
        .agg(pl.len().alias("rides"))
        .with_columns(
            pl.col("rides").truediv(pl.col("rides").sum().over(by)).mul(100).round(digits).alias("pct")
        )
        .sort(col, by)
    )
    if pivot:
        out = out.pivot(on=by, index=col, values="pct").sort(col)
    return out


profile(df)

column,dtype,nulls,pct_null,uniques
str,str,i64,f64,i64
"""ride_id""","""String""",0,0.0,6006904
"""rideable_type""","""String""",0,0.0,2
"""started_at""","""Datetime(time_unit='us', time_…",0,0.0,6005573
"""ended_at""","""Datetime(time_unit='us', time_…",0,0.0,6003761
"""start_station_name""","""String""",1201906,20.0,1974
…,…,…,…,…
"""distance_km""","""Float64""",0,0.0,387532
"""start_inside_chicago""","""Boolean""",0,0.0,2
"""end_inside_chicago""","""Boolean""",0,0.0,2


In [9]:
summarize_by(df, "member_casual", share=True, station_nulls=True)

member_casual,rides_pct,ride_length_avg,ride_length_med,ride_length_std,distance_km_avg,distance_km_med,distance_km_std,start_station_name_null_pct,end_station_name_null_pct
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""casual""",35.128262,18.13,11.13,34.7,2.21,1.66,1.99,20.46,23.15
"""member""",64.871738,12.16,8.68,21.2,2.27,1.65,1.97,19.76,19.42


In [10]:
summarize_by(df, "rideable_type", share=True, station_nulls=True)

rideable_type,rides_pct,ride_length_avg,ride_length_med,ride_length_std,distance_km_avg,distance_km_med,distance_km_std,start_station_name_null_pct,end_station_name_null_pct
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""classic_bike""",29.612193,18.35,10.67,44.52,2.02,1.44,1.9,0.0,0.0
"""electric_bike""",70.387807,12.53,9.03,13.52,2.34,1.76,2.0,28.43,29.45


In [11]:
summarize_by(df, "day_of_week", share=True, station_nulls=True)

day_of_week,rides_pct,ride_length_avg,ride_length_med,ride_length_std,distance_km_avg,distance_km_med,distance_km_std,start_station_name_null_pct,end_station_name_null_pct
i8,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,13.097962,13.84,9.03,26.19,2.2,1.61,1.95,18.64,18.94
2,14.471115,12.83,8.85,23.37,2.22,1.64,1.93,18.99,19.71
3,14.616731,12.56,8.8,23.29,2.22,1.64,1.9,19.72,20.43
4,14.618346,12.93,8.88,23.84,2.23,1.64,1.93,20.11,20.84
5,15.043923,14.21,9.35,28.36,2.21,1.64,1.95,20.13,21.06
6,15.720161,16.77,11.0,29.96,2.36,1.77,2.09,21.59,22.65
7,12.431762,16.78,10.63,31.9,2.29,1.7,2.07,20.72,21.2


In [12]:
summarize_by(df, "season", share=True, station_nulls=True)

season,rides_pct,ride_length_avg,ride_length_med,ride_length_std,distance_km_avg,distance_km_med,distance_km_std,start_station_name_null_pct,end_station_name_null_pct
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Autumn""",28.739664,13.89,9.15,27.56,2.23,1.64,1.96,20.6,21.59
"""Spring""",22.961113,13.95,9.08,26.36,2.21,1.62,1.97,19.27,19.76
"""Summer""",40.516662,15.19,10.32,25.52,2.35,1.77,2.03,20.68,21.2
"""Winter""",7.782562,11.68,7.42,32.07,1.88,1.38,1.68,16.49,17.97


In [13]:
summarize_by(df, "month", share=True, station_nulls=True)

month,rides_pct,ride_length_avg,ride_length_med,ride_length_std,distance_km_avg,distance_km_med,distance_km_std,start_station_name_null_pct,end_station_name_null_pct
i8,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,2.226288,11.83,7.15,40.25,1.82,1.35,1.62,16.63,18.41
2,3.284587,11.42,7.57,22.16,1.96,1.41,1.75,15.28,16.36
3,5.134742,12.84,8.33,25.02,2.08,1.52,1.86,17.33,18.57
4,7.249292,13.02,8.67,23.54,2.17,1.58,1.94,19.44,19.61
5,10.577079,15.12,9.82,28.67,2.3,1.69,2.05,20.09,20.44
…,…,…,…,…,…,…,…,…,…
8,14.091802,14.88,10.18,23.86,2.35,1.77,2.03,21.11,21.81
9,12.507308,14.27,9.63,23.93,2.31,1.71,2.01,20.12,21.08
10,10.462311,14.14,9.2,29.4,2.23,1.64,1.96,21.39,22.3


In [14]:
for col in ("start_hour", "day_of_week", "month", "season"):
    print(col)
    display(distribution_by(df, "member_casual", col))

start_hour


start_hour,casual,member
i8,f64,f64
0,2.2,0.95
1,1.44,0.58
2,0.9,0.34
3,0.53,0.22
4,0.42,0.25
…,…,…
19,6.15,5.87
20,4.58,4.15
21,3.98,3.28


day_of_week


day_of_week,casual,member
i8,f64,f64
1,11.51,13.96
2,11.45,16.11
3,11.69,16.2
4,12.47,15.78
5,15.69,14.69
6,21.19,12.76
7,15.99,10.5


month


month,casual,member
i8,f64,f64
1,1.13,2.82
2,1.9,4.03
3,4.02,5.74
4,6.02,7.92
5,11.18,10.25
…,…,…
8,15.9,13.11
9,13.47,11.99
10,10.16,10.63


season


season,casual,member
str,f64,f64
"""Autumn""",28.11,29.08
"""Spring""",21.21,23.91
"""Summer""",46.36,37.35
"""Winter""",4.31,9.66


In [15]:
(
    df.filter(pl.col("is_round_trip").is_not_null())
    .group_by("member_casual", "is_round_trip")
    .agg(pl.len().alias("total_rides"))
    .with_columns(
        (
            pl.col("total_rides")
            / pl.col("total_rides").sum().over("member_casual")
            * 100
        ).round(2).alias("percentage")
    )
    .sort("total_rides", descending=True)
)

member_casual,is_round_trip,total_rides,percentage
str,bool,u32,f64
"""member""",false,2594419,97.79
"""casual""",false,1292988,92.59
"""casual""",true,103479,7.41
"""member""",true,58540,2.21


In [16]:
(
    df.select(STATION_COLS)
    .unpivot(value_name="station")
    .filter(pl.col("station").is_not_null())
    .group_by("station")
    .agg(pl.len().alias("total_rides"))
    .top_k(10, by="total_rides")
    .sort("total_rides", descending=True)
)

station,total_rides
str,u32
"""Navy Pier""",130508
"""DuSable Lake Shore Dr & Monroe…",70877
"""Michigan Ave & Oak St""",68171
"""DuSable Lake Shore Dr & North …",63194
"""State St & Chicago Ave""",60206
"""Canal St & Madison St""",57926
"""Wells St & Concord Ln""",56659
"""Wells St & Elm St""",54200
"""Theater on the Lake""",53048
